# Political Reddit Scraper Driven by News API

In [5]:
%pip install praw pandas requests

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
from datetime import datetime, timezone
from itertools import islice
import time

import pandas as pd
import praw
import requests



## Step 2: Configure APIs and Collection Targets
The notebook uses your **NewsData.io** API key to pull recent political news, turns those headlines into search queries, then searches multiple political subreddits for related Reddit discussions.

In [7]:
REDDIT_CLIENT_ID = "ANAhS6ga7S-dXxS7mxsn7w"
REDDIT_CLIENT_SECRET = "Dwowr2WXUgqGDJkHbOI2Sx1Jws7DfA"
REDDIT_USER_AGENT = "social_analytics_project/1.0"
NEWS_API_KEY = "pub_b3c903e5cc6c47f791e8190a0cd7850e"

In [8]:
POLITICAL_SUBREDDITS = [
    "politics",
    "PoliticalDiscussion",
    "Conservative",
    "Liberal",
    "democrats",
    "Republican",
    "worldnews",
    "news",
]

TARGET_POSTS = 220
COMMENTS_PER_POST = 20
NEWS_ARTICLE_LIMIT = 12
SEARCH_LIMIT_PER_QUERY = 40
REQUEST_DELAY_SECONDS = 1

In [9]:
reddit = praw.Reddit(
    client_id=REDDIT_CLIENT_ID,
    client_secret=REDDIT_CLIENT_SECRET,
    user_agent=REDDIT_USER_AGENT,
    check_for_async=False,
    ratelimit_seconds=120,
    timeout=30,
    read_only=True,
)

print(f"Read-only mode: {reddit.read_only}")
print(f"Target post count: {TARGET_POSTS}")
print(f"Political subreddits: {', '.join(POLITICAL_SUBREDDITS)}")

Read-only mode: True
Target post count: 220
Political subreddits: politics, PoliticalDiscussion, Conservative, Liberal, democrats, Republican, worldnews, news


## Step 3: Collect More Than 200 Political Posts and Their Comments
This step fetches current political headlines from the news API, builds Reddit search queries from those headlines, searches several political subreddits, and saves both post-level data and comment-level data. The collection stops once it reaches **more than 200 unique posts**.

In [10]:
def utc_string(timestamp):
    return datetime.fromtimestamp(timestamp, tz=timezone.utc).strftime("%Y-%m-%d %H:%M:%S")


def normalize_query(text, max_words=10, max_chars=80):
    cleaned = " ".join((text or "").replace("|", " ").split())
    if not cleaned:
        return ""
    shortened = " ".join(cleaned.split()[:max_words])
    return shortened[:max_chars].strip()


def fetch_news_contexts(api_key, article_limit=12):
    url = "https://newsdata.io/api/1/news"
    params = {
        "apikey": api_key,
        "category": "politics",
        "language": "en",
    }

    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()
    payload = response.json()
    results = payload.get("results", [])[:article_limit]

    if not results:
        raise ValueError("The news API did not return any political articles.")

    news_contexts = []
    seen_queries = set()

    for article in results:
        title = (article.get("title") or "").strip()
        description = (article.get("description") or "").strip()
        source_name = (article.get("source_id") or article.get("source_name") or "unknown").strip()
        article_url = article.get("link") or ""

        title_query = normalize_query(title)
        description_query = normalize_query(description, max_words=8, max_chars=60)

        for query in [title_query, description_query]:
            if query and query.lower() not in seen_queries:
                seen_queries.add(query.lower())
                news_contexts.append({
                    "search_query": query,
                    "news_title": title,
                    "news_description": description,
                    "news_source": source_name,
                    "news_url": article_url,
                })

    return news_contexts

In [11]:
news_contexts = fetch_news_contexts(NEWS_API_KEY, article_limit=NEWS_ARTICLE_LIMIT)
print(f"News-driven queries prepared: {len(news_contexts)}")
print(pd.DataFrame(news_contexts).head(10))

News-driven queries prepared: 20
                                        search_query  \
0  Riverside Brookfield High School students walk...   
1  Roughly 150 Riverside Brookfield High School s...   
2           The girl in the frame and an aunt in the   
3                  The girl in the frame and an aunt   
4  Postal ballots favored Jamaat despite BNP winn...   
5       As a single party, Jamaat received 45.88% of   
6  PM in waiting Balen Shah defeats Oli in Jhapa,...   
7          Balen Shah defeats former PM K. P. Sharma   
8  Billions Spent, But Where Are The Fighters? Ir...   
9  Despite years of investment to prepare these p...   

                                          news_title  \
0  Riverside Brookfield High School students walk...   
1  Riverside Brookfield High School students walk...   
2  The girl in the frame and an aunt in the shado...   
3  The girl in the frame and an aunt in the shado...   
4  Postal ballots favored Jamaat despite BNP winn...   
5  Postal ball

In [12]:
posts_data = []
comments_data = []
seen_post_ids = set()

In [13]:
for subreddit_name in POLITICAL_SUBREDDITS:
    subreddit = reddit.subreddit(subreddit_name)
    print(f"Searching r/{subreddit_name} ...")

    for context in news_contexts:
        query = context["search_query"]

        try:
            submissions = subreddit.search(
                query,
                sort="new",
                time_filter="year",
                limit=SEARCH_LIMIT_PER_QUERY,
            )

            for post in submissions:
                if post.id in seen_post_ids:
                    continue

                seen_post_ids.add(post.id)
                post.comments.replace_more(limit=0)
                selected_comments = list(islice(post.comments.list(), COMMENTS_PER_POST))

                posts_data.append({
                    "post_id": post.id,
                    "title": post.title,
                    "selftext": post.selftext,
                    "score": post.score,
                    "num_comments": post.num_comments,
                    "upvote_ratio": getattr(post, "upvote_ratio", None),
                    "subreddit": str(post.subreddit),
                    "author": str(post.author),
                    "post_url": f"https://www.reddit.com{post.permalink}",
                    "external_url": post.url,
                    "search_query": query,
                    "matched_news_title": context["news_title"],
                    "matched_news_source": context["news_source"],
                    "matched_news_url": context["news_url"],
                    "created_utc": utc_string(post.created_utc),
                })

                for comment in selected_comments:
                    comments_data.append({
                        "comment_id": comment.id,
                        "post_id": post.id,
                        "subreddit": str(post.subreddit),
                        "post_title": post.title,
                        "comment_author": str(comment.author),
                        "comment_body": comment.body,
                        "comment_score": comment.score,
                        "comment_depth": comment.depth,
                        "comment_created_utc": utc_string(comment.created_utc),
                        "search_query": query,
                    })

                if len(posts_data) >= TARGET_POSTS:
                    break

            if len(posts_data) >= TARGET_POSTS:
                break

            time.sleep(REQUEST_DELAY_SECONDS)

        except Exception as exc:
            print(f"Skipped query '{query}' in r/{subreddit_name}: {exc}")

    if len(posts_data) >= TARGET_POSTS:
        break

Searching r/politics ...
Searching r/PoliticalDiscussion ...
Searching r/Conservative ...


In [14]:
posts_df = pd.DataFrame(posts_data)
comments_df = pd.DataFrame(comments_data)

if posts_df.empty:
    raise ValueError("No Reddit posts were collected. Re-run the cell or widen the subreddit/query settings.")

posts_df = posts_df.sort_values("created_utc", ascending=False).reset_index(drop=True)
comments_df = comments_df.sort_values("comment_created_utc", ascending=False).reset_index(drop=True)

print(f"Total unique posts collected: {len(posts_df)}")
print(f"Total comments collected: {len(comments_df)}")
print(posts_df[["subreddit", "search_query", "title"]].head(10))

Total unique posts collected: 220
Total comments collected: 3796
             subreddit                                       search_query  \
0  PoliticalDiscussion  Despite years of investment to prepare these p...   
1  PoliticalDiscussion  Israel issues new evacuation orders over strik...   
2             politics       US and Israeli attacks on Iran are depleting   
3             politics  Riverside Brookfield High School students walk...   
4             politics       US and Israeli attacks on Iran are depleting   
5             politics       US and Israeli attacks on Iran are depleting   
6             politics       US and Israeli attacks on Iran are depleting   
7             politics       US and Israeli attacks on Iran are depleting   
8             politics       US and Israeli attacks on Iran are depleting   
9             politics       US and Israeli attacks on Iran are depleting   

                                               title  
0  Will Gulf states reconsider t

## Step 4: Review and Export the Dataset


In [15]:
print(f"Posts dataset shape: {posts_df.shape}")
print(f"Comments dataset shape: {comments_df.shape}")
print(f"Reached more than 200 posts: {len(posts_df) > 200}")

display(posts_df.head(5))
display(comments_df[["post_id", "comment_author", "comment_body", "comment_score"]].head(5))

Posts dataset shape: (220, 15)
Comments dataset shape: (3796, 10)
Reached more than 200 posts: True


,post_id,title,selftext,score,num_comments,upvote_ratio,subreddit,author,post_url,external_url,search_query,matched_news_title,matched_news_source,matched_news_url,created_utc
0,1rnbgos,Will Gulf states reconsider their investment p...,"The war involving Israel, the United States, a...",2,7,0.63,PoliticalDiscussion,Only-Deal-881,https://www.reddit.com/r/PoliticalDiscussion/c...,https://www.reddit.com/r/PoliticalDiscussion/c...,Despite years of investment to prepare these p...,"Billions Spent, But Where Are The Fighters? Ir...",timesnownews,https://www.timesnownews.com/world/middle-east...,2026-03-07 14:34:32
1,1rn20s6,How long will the world tolerate double standa...,Around the world people are growing tired of t...,0,11,0.14,PoliticalDiscussion,zelotakelazam,https://www.reddit.com/r/PoliticalDiscussion/c...,https://www.reddit.com/r/PoliticalDiscussion/c...,Israel issues new evacuation orders over strik...,Israel issues new evacuation orders over strik...,naharnet,https://www.naharnet.com/stories/en/318805-isr...,2026-03-07 05:55:53
2,1rln5hb,The US and Israel are waging war on an Iran th...,,29,8,0.79,politics,drtolmn69,https://www.reddit.com/r/politics/comments/1rl...,https://www.theguardian.com/commentisfree/2026...,US and Israeli attacks on Iran are depleting,Trump administration and Democrats at odds ove...,economictimes_indiatimes,https://economictimes.indiatimes.com/news/defe...,2026-03-05 17:08:53
3,1rktvro,AMA: Brandt Robinson a teacher running for U.S...,"Hi, I’m Brandt Robinson, a 29-year History tea...",45,36,0.84,politics,BrandtForCongress,https://www.reddit.com/r/politics/comments/1rk...,https://www.reddit.com/r/politics/comments/1rk...,Riverside Brookfield High School students walk...,Riverside Brookfield High School students walk...,rblandmark,https://www.rblandmark.com/2026/03/07/riversid...,2026-03-04 18:48:13
4,1rka6oo,Trump news at a glance: Rubio and his boss can...,,107,12,0.92,politics,Pixiefairy2525,https://www.reddit.com/r/politics/comments/1rk...,https://www.theguardian.com/us-news/2026/mar/0...,US and Israeli attacks on Iran are depleting,Trump administration and Democrats at odds ove...,economictimes_indiatimes,https://economictimes.indiatimes.com/news/defe...,2026-03-04 03:06:52


,post_id,comment_author,comment_body,comment_score
0,1rn20s6,reaper527,> This raises a simple question: if internatio...,1
1,1rn20s6,Horror_Adventurous,International law is nothing without anyone to...,1
2,1rn20s6,OmOshIroIdEs,The funniest part is your reference to the ICC...,1
3,1rnbgos,WhatAreYouSaying05,The “investments” were bullshit anyway. They k...,1
4,1rn20s6,Soepoelse123,To paraphrase Bull (1982): until everyone is s...,1


In [16]:
posts_summary = posts_df.groupby("subreddit").size().sort_values(ascending=False).rename("post_count")
comments_summary = comments_df.groupby("subreddit").size().sort_values(ascending=False).rename("comment_count")

print("Posts by subreddit:")
display(posts_summary.to_frame())

print("Comments by subreddit:")
display(comments_summary.to_frame())

Posts by subreddit:


,post_count
subreddit,
politics,104
PoliticalDiscussion,92
Conservative,24


Comments by subreddit:


,comment_count
subreddit,
politics,1805
PoliticalDiscussion,1723
Conservative,268


In [17]:
posts_output_path = "reddit_political_posts.csv"
comments_output_path = "reddit_political_comments.csv"

posts_df.to_csv(posts_output_path, index=False, encoding="utf-8-sig")
comments_df.to_csv(comments_output_path, index=False, encoding="utf-8-sig")

print(f"Saved posts to: {posts_output_path}")
print(f"Saved comments to: {comments_output_path}")

Saved posts to: reddit_political_posts.csv
Saved comments to: reddit_political_comments.csv


In [18]:
posts_df[[
    "post_id",
    "subreddit",
    "search_query",
    "matched_news_title",
    "title",
    "num_comments",
    "created_utc",
]].head(15)

,post_id,subreddit,search_query,matched_news_title,title,num_comments,created_utc
0,1rnbgos,PoliticalDiscussion,Despite years of investment to prepare these p...,"Billions Spent, But Where Are The Fighters? Ir...",Will Gulf states reconsider their investment p...,7,2026-03-07 14:34:32
1,1rn20s6,PoliticalDiscussion,Israel issues new evacuation orders over strik...,Israel issues new evacuation orders over strik...,How long will the world tolerate double standa...,11,2026-03-07 05:55:53
2,1rln5hb,politics,US and Israeli attacks on Iran are depleting,Trump administration and Democrats at odds ove...,The US and Israel are waging war on an Iran th...,8,2026-03-05 17:08:53
3,1rktvro,politics,Riverside Brookfield High School students walk...,Riverside Brookfield High School students walk...,AMA: Brandt Robinson a teacher running for U.S...,36,2026-03-04 18:48:13
4,1rka6oo,politics,US and Israeli attacks on Iran are depleting,Trump administration and Democrats at odds ove...,Trump news at a glance: Rubio and his boss can...,12,2026-03-04 03:06:52
5,1rjjgr5,politics,US and Israeli attacks on Iran are depleting,Trump administration and Democrats at odds ove...,Democrats’ newfound unity faces a test after U...,61,2026-03-03 07:56:16
6,1rj5uoq,politics,US and Israeli attacks on Iran are depleting,Trump administration and Democrats at odds ove...,US and Israeli interests may soon diverge on Iran,4,2026-03-02 21:29:00
7,1rir3fm,politics,US and Israeli attacks on Iran are depleting,Trump administration and Democrats at odds ove...,Where things stand after the US and Israeli st...,5,2026-03-02 12:07:17
8,1ri9o86,politics,US and Israeli attacks on Iran are depleting,Trump administration and Democrats at odds ove...,3 US troops killed and 5 are seriously wounded...,22,2026-03-01 21:28:16
9,1ri4jyt,politics,US and Israeli attacks on Iran are depleting,Trump administration and Democrats at odds ove...,World leaders react to US and Israeli strikes ...,7,2026-03-01 18:17:33


## Step 5: Text Preprocessing and Sentiment Analysis
Now we'll clean and analyze the collected Reddit posts and comments using NLP techniques.

In [19]:
%pip install nltk emoji textblob

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [20]:
import warnings
warnings.filterwarnings("ignore")

import re
import string
import emoji
import nltk
from nltk.corpus import stopwords
from textblob import TextBlob

# Download required NLTK data
nltk.download('stopwords', quiet=True)
STOPWORDS = set(stopwords.words('english'))

print("Text processing libraries loaded successfully")

Text processing libraries loaded successfully


In [21]:
def remove_urls(text):
    return re.sub(r'https?://\S+|www\.\S+', '', text)

def remove_html(text):
    return re.sub(r'<.*?>', '', text)

def remove_numbers(text):
    return re.sub(r'\d+', '', text)

def remove_hashtags(text):
    return re.sub(r'#\w+', '', text)

def remove_mentions(text):
    return re.sub(r'@\w+', '', text)

def remove_punctuation(text):
    return text.translate(str.maketrans('', '', string.punctuation))

def remove_stopwords(text):
    return " ".join([word for word in text.split() if word not in STOPWORDS])

def preprocess_text(text):
    """Clean and preprocess text for analysis"""
    if not isinstance(text, str) or not text.strip():
        return ""
    
    text = emoji.demojize(text)
    text = remove_urls(text)
    text = remove_html(text)
    text = remove_numbers(text)
    text = remove_hashtags(text)
    text = remove_mentions(text)
    text = remove_punctuation(text)
    text = text.lower()
    text = remove_stopwords(text)
    return text.strip()

def analyze_sentiment(text):
    """Classify text sentiment as positive, negative, or neutral"""
    if not text or not isinstance(text, str):
        return "neutral"
    
    score = TextBlob(text).sentiment.polarity
    if score > 0.05:
        return "positive"
    elif score < -0.05:
        return "negative"
    else:
        return "neutral"

print("Preprocessing and sentiment analysis functions defined")

Preprocessing and sentiment analysis functions defined


In [22]:
# Preprocess post titles and text
posts_df["title_clean"] = posts_df["title"].apply(preprocess_text)
posts_df["selftext_clean"] = posts_df["selftext"].apply(preprocess_text)

# Combine title and selftext for complete post analysis
posts_df["full_text_clean"] = posts_df["title_clean"] + " " + posts_df["selftext_clean"]
posts_df["full_text_clean"] = posts_df["full_text_clean"].str.strip()

print(f"Preprocessed {len(posts_df)} posts")
print(f"Average cleaned post length: {posts_df['full_text_clean'].str.len().mean():.0f} characters")

# Show example
display(posts_df[["title", "title_clean", "full_text_clean"]].head(3))

Preprocessed 220 posts
Average cleaned post length: 1074 characters


,title,title_clean,full_text_clean
0,Will Gulf states reconsider their investment p...,gulf states reconsider investment plans demand...,gulf states reconsider investment plans demand...
1,How long will the world tolerate double standa...,long world tolerate double standards war,long world tolerate double standards war aroun...
2,The US and Israel are waging war on an Iran th...,us israel waging war iran think know reality d...,us israel waging war iran think know reality d...


In [23]:
# Preprocess comments
comments_df["comment_clean"] = comments_df["comment_body"].apply(preprocess_text)

print(f"Preprocessed {len(comments_df)} comments")
print(f"Average cleaned comment length: {comments_df['comment_clean'].str.len().mean():.0f} characters")

# Show example
display(comments_df[["comment_body", "comment_clean"]].head(3))

Preprocessed 3796 comments
Average cleaned comment length: 236 characters


,comment_body,comment_clean
0,> This raises a simple question: if internatio...,raises simple question international rules mat...
1,International law is nothing without anyone to...,international law nothing without anyone enfor...
2,The funniest part is your reference to the ICC...,funniest part reference icc guess people see “...


In [24]:
# Perform sentiment analysis on posts
posts_df["sentiment"] = posts_df["full_text_clean"].apply(analyze_sentiment)

print("Post sentiment distribution:")
print(posts_df["sentiment"].value_counts())
print(f"\nPercentages:")
print(posts_df["sentiment"].value_counts(normalize=True) * 100)

# Show examples of each sentiment
print("\nExample posts by sentiment:")
for sentiment_type in ["positive", "negative", "neutral"]:
    example = posts_df[posts_df["sentiment"] == sentiment_type].iloc[0] if len(posts_df[posts_df["sentiment"] == sentiment_type]) > 0 else None
    if example is not None:
        print(f"\n{sentiment_type.upper()}: {example['title'][:100]}...")

Post sentiment distribution:
sentiment
positive    129
neutral      77
negative     14
Name: count, dtype: int64

Percentages:
sentiment
positive    58.636364
neutral     35.000000
negative     6.363636
Name: proportion, dtype: float64

Example posts by sentiment:

POSITIVE: Will Gulf states reconsider their investment plans or demand compensation from the US?...

NEGATIVE: 3 US troops killed and 5 are seriously wounded during Iran attacks, military says...

NEUTRAL: How long will the world tolerate double standards in war?...


In [25]:
# Perform sentiment analysis on comments
comments_df["sentiment"] = comments_df["comment_clean"].apply(analyze_sentiment)

print("Comment sentiment distribution:")
print(comments_df["sentiment"].value_counts())
print(f"\nPercentages:")
print(comments_df["sentiment"].value_counts(normalize=True) * 100)

# Show examples
print("\nExample comments by sentiment:")
for sentiment_type in ["positive", "negative", "neutral"]:
    example = comments_df[comments_df["sentiment"] == sentiment_type].iloc[0] if len(comments_df[comments_df["sentiment"] == sentiment_type]) > 0 else None
    if example is not None:
        print(f"\n{sentiment_type.upper()}: {example['comment_body'][:150]}...")

Comment sentiment distribution:
sentiment
positive    1573
neutral     1385
negative     838
Name: count, dtype: int64

Percentages:
sentiment
positive    41.438356
neutral     36.485774
negative    22.075869
Name: proportion, dtype: float64

Example comments by sentiment:

POSITIVE: International law is nothing without anyone to enforce it. And the only way to enforce it is either through war or very good leverage. Certain countri...

NEGATIVE: > This raises a simple question: if international rules matter, shouldn’t they apply to everyone equally?

that's a flawed question, because internati...

NEUTRAL: The funniest part is your reference to the ICC. I guess people see “International” and think it’s some sort of a world tribunal. In fact, neither the ...


In [26]:
# Sentiment by subreddit for posts
post_sentiment_by_sub = pd.crosstab(
    posts_df["subreddit"], 
    posts_df["sentiment"], 
    normalize="index"
) * 100

print("Post sentiment distribution by subreddit (%):")
display(post_sentiment_by_sub.round(1))

# Add total post counts
post_counts = posts_df.groupby("subreddit").size().rename("total_posts")
post_summary = post_sentiment_by_sub.join(post_counts)
display(post_summary)

Post sentiment distribution by subreddit (%):


sentiment,negative,neutral,positive
subreddit,,,
Conservative,8.3,37.5,54.2
PoliticalDiscussion,4.3,37.0,58.7
politics,7.7,32.7,59.6


,negative,neutral,positive,total_posts
subreddit,,,,
Conservative,8.333333,37.500000,54.166667,24
PoliticalDiscussion,4.347826,36.956522,58.695652,92
politics,7.692308,32.692308,59.615385,104


In [27]:
# Sentiment by subreddit for comments
comment_sentiment_by_sub = pd.crosstab(
    comments_df["subreddit"], 
    comments_df["sentiment"], 
    normalize="index"
) * 100

print("Comment sentiment distribution by subreddit (%):")
display(comment_sentiment_by_sub.round(1))

# Add total comment counts
comment_counts = comments_df.groupby("subreddit").size().rename("total_comments")
comment_summary = comment_sentiment_by_sub.join(comment_counts)
display(comment_summary)

Comment sentiment distribution by subreddit (%):


sentiment,negative,neutral,positive
subreddit,,,
Conservative,18.3,44.8,36.9
PoliticalDiscussion,20.5,31.9,47.6
politics,24.1,39.7,36.2


,negative,neutral,positive,total_comments
subreddit,,,,
Conservative,18.283582,44.776119,36.940299,268
PoliticalDiscussion,20.545560,31.863030,47.591410,1723
politics,24.099723,39.667590,36.232687,1805


In [28]:
# Analyze how sentiment correlates with engagement
print("Average engagement metrics by post sentiment:")
engagement_by_sentiment = posts_df.groupby("sentiment").agg({
    "score": "mean",
    "num_comments": "mean",
    "upvote_ratio": "mean"
}).round(2)

display(engagement_by_sentiment)

# Analyze comment sentiment by post sentiment
print("\n\nComment sentiment vs Post sentiment:")
comment_post_sentiment = comments_df.merge(
    posts_df[["post_id", "sentiment"]], 
    on="post_id", 
    suffixes=("_comment", "_post")
)

sentiment_matrix = pd.crosstab(
    comment_post_sentiment["sentiment_post"],
    comment_post_sentiment["sentiment_comment"],
    normalize="index"
) * 100

print("Comment sentiment distribution (%) by post sentiment:")
display(sentiment_matrix.round(1))

Average engagement metrics by post sentiment:


,score,num_comments,upvote_ratio
sentiment,,,
negative,4410.50,899.64,0.87
neutral,1003.81,272.58,0.83
positive,1051.22,621.99,0.84




Comment sentiment vs Post sentiment:
Comment sentiment distribution (%) by post sentiment:


sentiment_comment,negative,neutral,positive
sentiment_post,,,
negative,25.7,39.6,34.7
neutral,23.3,35.0,41.6
positive,21.0,37.0,42.0


## Step 6: Export Enhanced Dataset with Sentiment Analysis

In [29]:
# Export enhanced datasets with sentiment analysis and cleaned text
posts_enhanced_path = "reddit_political_posts_enhanced.csv"
comments_enhanced_path = "reddit_political_comments_enhanced.csv"

posts_df.to_csv(posts_enhanced_path, index=False, encoding="utf-8-sig")
comments_df.to_csv(comments_enhanced_path, index=False, encoding="utf-8-sig")

print(f"✓ Saved enhanced posts to: {posts_enhanced_path}")
print(f"✓ Saved enhanced comments to: {comments_enhanced_path}")
print(f"\nNew columns added:")
print("Posts: title_clean, selftext_clean, full_text_clean, sentiment")
print("Comments: comment_clean, sentiment")
print(f"\nFinal dataset summary:")
print(f"- {len(posts_df)} posts with {posts_df['sentiment'].value_counts().to_dict()} sentiment distribution")
print(f"- {len(comments_df)} comments with {comments_df['sentiment'].value_counts().to_dict()} sentiment distribution")

✓ Saved enhanced posts to: reddit_political_posts_enhanced.csv
✓ Saved enhanced comments to: reddit_political_comments_enhanced.csv

New columns added:
Posts: title_clean, selftext_clean, full_text_clean, sentiment
Comments: comment_clean, sentiment

Final dataset summary:
- 220 posts with {'positive': 129, 'neutral': 77, 'negative': 14} sentiment distribution
- 3796 comments with {'positive': 1573, 'neutral': 1385, 'negative': 838} sentiment distribution


In [30]:
# Preview the enhanced datasets
print("Enhanced Posts Dataset Sample:")
display(posts_df[[
    "subreddit", 
    "title", 
    "sentiment", 
    "score", 
    "num_comments",
    "matched_news_title"
]].head(10))

print("\n\nEnhanced Comments Dataset Sample:")
display(comments_df[[
    "subreddit",
    "comment_body",
    "sentiment",
    "comment_score"
]].head(10))

Enhanced Posts Dataset Sample:


,subreddit,title,sentiment,score,num_comments,matched_news_title
0,PoliticalDiscussion,Will Gulf states reconsider their investment p...,positive,2,7,"Billions Spent, But Where Are The Fighters? Ir..."
1,PoliticalDiscussion,How long will the world tolerate double standa...,neutral,0,11,Israel issues new evacuation orders over strik...
2,politics,The US and Israel are waging war on an Iran th...,neutral,29,8,Trump administration and Democrats at odds ove...
3,politics,AMA: Brandt Robinson a teacher running for U.S...,neutral,45,36,Riverside Brookfield High School students walk...
4,politics,Trump news at a glance: Rubio and his boss can...,neutral,107,12,Trump administration and Democrats at odds ove...
5,politics,Democrats’ newfound unity faces a test after U...,neutral,0,61,Trump administration and Democrats at odds ove...
6,politics,US and Israeli interests may soon diverge on Iran,neutral,0,4,Trump administration and Democrats at odds ove...
7,politics,Where things stand after the US and Israeli st...,neutral,17,5,Trump administration and Democrats at odds ove...
8,politics,3 US troops killed and 5 are seriously wounded...,negative,135,22,Trump administration and Democrats at odds ove...
9,politics,World leaders react to US and Israeli strikes ...,neutral,39,7,Trump administration and Democrats at odds ove...




Enhanced Comments Dataset Sample:


,subreddit,comment_body,sentiment,comment_score
0,PoliticalDiscussion,> This raises a simple question: if internatio...,negative,1
1,PoliticalDiscussion,International law is nothing without anyone to...,positive,1
2,PoliticalDiscussion,The funniest part is your reference to the ICC...,neutral,1
3,PoliticalDiscussion,The “investments” were bullshit anyway. They k...,neutral,1
4,PoliticalDiscussion,To paraphrase Bull (1982): until everyone is s...,neutral,1
5,PoliticalDiscussion,Say what?,neutral,1
6,PoliticalDiscussion,"""The World""? ""The World"" has no teeth. The USA...",neutral,1
7,PoliticalDiscussion,International law isn't anything that nations ...,positive,1
8,PoliticalDiscussion,…you believe that there is still trust in the ...,neutral,1
9,PoliticalDiscussion,The US has been doing shit like this for 150 y...,negative,1
